# Code-Only System Prompt Comparison

This notebook compares the previous corrected baseline results with the new runs that add one system prompt:

> Output only the code, no explanations.

Comparison groups:

- **baseline_fixed**: `results/*_fixextractor.jsonl`
- **codeonly_system**: `results_codeonly/*_1_50.jsonl`

The comparison is paired by `(model, problem_id, template_id)`, so each code-only run is compared with the matching previous run.

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
RESULTS_DIR = ROOT / "results"
CODEONLY_DIR = ROOT / "results_codeonly"
PROMPT_MATRIX_PATH = ROOT / "prompt_matrix_prompt_template.08.csv"
OUTPUT_DIR = ROOT / "analysis_outputs" / "codeonly_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 1. Load Results

The baseline and code-only files share the same `task_id` format: `problem_id_template_id`.

In [ ]:
def read_jsonl(path: Path, condition: str) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["source_file"] = path.name
            row["condition"] = condition
            rows.append(row)
    return pd.DataFrame(rows)


def parse_task_id(value: object) -> tuple[int, int]:
    match = re.fullmatch(r"(\d+)_(\d+)", str(value))
    if not match:
        raise ValueError(f"Unexpected task_id: {value!r}")
    return int(match.group(1)), int(match.group(2))


def model_from_filename(path: Path) -> str:
    name = path.name
    name = name.removesuffix("_fixextractor.jsonl")
    name = name.removesuffix("_1_50.jsonl")
    name = name.removesuffix("_final.jsonl")
    return name

baseline_paths = sorted(RESULTS_DIR.glob("*_fixextractor.jsonl"))
codeonly_paths = sorted(CODEONLY_DIR.glob("*.jsonl"))

baseline = pd.concat([read_jsonl(path, "baseline_fixed") for path in baseline_paths], ignore_index=True)
codeonly = pd.concat([read_jsonl(path, "codeonly_system") for path in codeonly_paths], ignore_index=True)

for frame in [baseline, codeonly]:
    parsed = frame["task_id"].map(parse_task_id)
    frame["problem_id"] = parsed.map(lambda x: x[0])
    frame["template_id"] = parsed.map(lambda x: x[1])
    frame["model_short"] = frame["source_file"].map(lambda x: model_from_filename(Path(x)))

prompt_matrix = pd.read_csv(PROMPT_MATRIX_PATH).rename(columns={"ID": "template_id", "Role framing": "Role_framing"})

df = pd.concat([baseline, codeonly], ignore_index=True).merge(prompt_matrix, on="template_id", how="left", validate="many_to_one")

print("baseline files:", [p.name for p in baseline_paths])
print("code-only files:", [p.name for p in codeonly_paths])
print("rows:", len(df))
print(df.groupby(["condition", "model_short"]).size().to_string())
df.head(3)

## 2. Derived Metrics

We compute the same performance and behavior metrics used in the previous analysis.

In [ ]:
def safe_len(value: object) -> int:
    if isinstance(value, str):
        return len(value)
    if pd.isna(value):
        return 0
    return len(str(value))


def error_text(errors: object) -> str:
    if isinstance(errors, list):
        return "\n".join(map(str, errors))
    if pd.isna(errors):
        return ""
    return str(errors)


def classify_error(errors: object, extracted_code: object) -> str:
    text = error_text(errors).lower()
    code = extracted_code if isinstance(extracted_code, str) else ""
    if "response is not valid python" in text:
        return "invalid_python_response"
    if not code.strip():
        return "missing_or_empty_code"
    if not text.strip():
        return "none"
    if "not defined" in text:
        return "name_not_defined"
    if "syntax" in text or "invalid syntax" in text:
        return "syntax_error"
    if "timeout" in text or "timed out" in text:
        return "timeout"
    if "importerror" in text or "no module named" in text:
        return "import_error"
    if "assert" in text or "test failed" in text:
        return "wrong_answer"
    return "other_runtime"


df["passed_int"] = df["passed"].astype(bool).astype(int)
df["tests_passed"] = pd.to_numeric(df["tests_passed"], errors="coerce")
df["tests_total"] = pd.to_numeric(df["tests_total"], errors="coerce")
df["tests_fraction"] = np.where(df["tests_total"] > 0, df["tests_passed"] / df["tests_total"], np.nan)
df["completion_tokens"] = pd.to_numeric(df["completion_tokens"], errors="coerce")
df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
df["prompt_tokens"] = pd.to_numeric(df["prompt_tokens"], errors="coerce")
df["response_chars"] = df["response_text"].map(safe_len)
df["code_chars"] = df["extracted_code"].map(safe_len)
df["has_extracted_code"] = (df["code_chars"] > 0).astype(int)
df["extra_text_chars"] = (df["response_chars"] - df["code_chars"]).clip(lower=0)
df["extra_text_ratio"] = np.where(df["response_chars"] > 0, df["extra_text_chars"] / df["response_chars"], np.nan)
df["likely_code_only"] = ((df["has_extracted_code"] == 1) & (df["extra_text_ratio"] < 0.15)).astype(int)
df["error_text"] = df["errors"].map(error_text)
df["error_category"] = [classify_error(e, c) for e, c in zip(df["errors"], df["extracted_code"])]
df["invalid_python"] = df["error_category"].eq("invalid_python_response").astype(int)

df.to_csv(OUTPUT_DIR / "joined_codeonly_comparison.csv", index=False)
df[["condition", "model_short", "problem_id", "template_id", "Tone", "passed_int", "tests_fraction", "completion_tokens", "likely_code_only", "error_category"]].head()

## 3. Overall Comparison

This table compares aggregate performance and behavior between the corrected baseline and the code-only system-prompt runs.

In [ ]:
summary = (
    df.groupby("condition")
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        passes=("passed_int", "sum"),
        tests_fraction=("tests_fraction", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        invalid_python_rate=("invalid_python", "mean"),
    )
    .reset_index()
)
summary.to_csv(OUTPUT_DIR / "overall_summary.csv", index=False)
summary.style.format({
    "pass_rate": "{:.1%}", "tests_fraction": "{:.1%}", "avg_completion_tokens": "{:.0f}",
    "avg_duration_seconds": "{:.2f}", "code_only_rate": "{:.1%}", "invalid_python_rate": "{:.1%}"
})

## 4. Model-Level Comparison

This shows whether the system prompt helps all models equally.

In [ ]:
model_summary = (
    df.groupby(["model_short", "condition"])
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        passes=("passed_int", "sum"),
        tests_fraction=("tests_fraction", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        invalid_python_rate=("invalid_python", "mean"),
    )
    .reset_index()
)
model_summary.to_csv(OUTPUT_DIR / "model_summary.csv", index=False)
model_summary.style.format({
    "pass_rate": "{:.1%}", "tests_fraction": "{:.1%}", "avg_completion_tokens": "{:.0f}",
    "avg_duration_seconds": "{:.2f}", "code_only_rate": "{:.1%}", "invalid_python_rate": "{:.1%}"
})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(
    axes,
    ["tests_fraction", "avg_completion_tokens", "code_only_rate"],
    ["Avg. tests passed", "Completion tokens", "Code-only rate"],
):
    sns.barplot(data=model_summary, x="model_short", y=metric, hue="condition", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 5. Paired Deltas Versus Baseline

Because the two conditions use the same `(model, problem_id, template_id)` rows, we can compute direct paired differences:

```text
codeonly_system - baseline_fixed
```

Positive performance deltas mean the code-only system prompt improved the metric. Negative token/runtime deltas mean it made outputs shorter/faster.

In [ ]:
metrics = ["passed_int", "tests_fraction", "completion_tokens", "duration_seconds", "likely_code_only", "extra_text_ratio", "invalid_python"]
keys = ["model_short", "problem_id", "template_id", "Tone", "Role_framing", "Format"]
wide = df.pivot_table(index=keys, columns="condition", values=metrics, aggfunc="first")
paired = wide.swaplevel(axis=1).sort_index(axis=1).reset_index()
for metric in metrics:
    paired[f"delta_{metric}"] = paired[("codeonly_system", metric)] - paired[("baseline_fixed", metric)]
paired.columns = ["_".join([str(x) for x in col if str(x) not in ["", "nan"]]).strip("_") if isinstance(col, tuple) else col for col in paired.columns]
paired.to_csv(OUTPUT_DIR / "paired_row_deltas.csv", index=False)

paired_summary = []
for metric in metrics:
    s = paired[f"delta_{metric}"].dropna()
    paired_summary.append({
        "metric": metric,
        "mean_delta": s.mean(),
        "median_delta": s.median(),
        "improved_count": int((s > 0).sum()),
        "worse_count": int((s < 0).sum()),
        "same_count": int((s == 0).sum()),
        "n": len(s),
    })
paired_summary = pd.DataFrame(paired_summary)
paired_summary.to_csv(OUTPUT_DIR / "paired_delta_summary.csv", index=False)
paired_summary.style.format({"mean_delta": "{:+.4f}", "median_delta": "{:+.4f}"})

## 6. Paired Deltas by Model and Tone

This shows where the system prompt helped or hurt.

In [ ]:
def summarize_deltas(group_cols):
    agg = {
        "delta_passed_int": ["mean", "sum"],
        "delta_tests_fraction": "mean",
        "delta_completion_tokens": "mean",
        "delta_duration_seconds": "mean",
        "delta_likely_code_only": "mean",
        "delta_extra_text_ratio": "mean",
        "delta_invalid_python": "mean",
    }
    out = paired.groupby(group_cols).agg(agg)
    out.columns = ["_".join(col).strip("_") for col in out.columns]
    return out.reset_index()

model_deltas = summarize_deltas(["model_short"])
tone_deltas = summarize_deltas(["Tone"])
model_tone_deltas = summarize_deltas(["model_short", "Tone"])

model_deltas.to_csv(OUTPUT_DIR / "model_deltas.csv", index=False)
tone_deltas.to_csv(OUTPUT_DIR / "tone_deltas.csv", index=False)
model_tone_deltas.to_csv(OUTPUT_DIR / "model_tone_deltas.csv", index=False)

print("Model deltas")
display(model_deltas.style.format({c: "{:+.4f}" for c in model_deltas.columns if c not in ["model_short"]}))
print("Tone deltas")
display(tone_deltas.style.format({c: "{:+.4f}" for c in tone_deltas.columns if c not in ["Tone"]}))
print("Model x tone deltas")
display(model_tone_deltas.style.format({c: "{:+.4f}" for c in model_tone_deltas.columns if c not in ["model_short", "Tone"]}))

## 7. Regression Check

This model estimates the average association of the code-only system prompt with performance/behavior while controlling for model, problem, and template.

```text
metric ~ condition + model fixed effects + problem fixed effects + template fixed effects
```

In [ ]:
regression_rows = []
for metric in metrics:
    formula = f"{metric} ~ C(condition) + C(model_short) + C(problem_id) + C(template_id)"
    model_data = df[[metric, "condition", "model_short", "problem_id", "template_id"]].dropna()
    result = smf.ols(formula, data=model_data).fit(cov_type="HC3")
    term = "C(condition)[T.codeonly_system]"
    regression_rows.append({
        "metric": metric,
        "coef_codeonly_vs_baseline": result.params.get(term, np.nan),
        "std_err": result.bse.get(term, np.nan),
        "p_value": result.pvalues.get(term, np.nan),
        "nobs": result.nobs,
        "rsquared": result.rsquared,
    })
regression_summary = pd.DataFrame(regression_rows)
regression_summary.to_csv(OUTPUT_DIR / "condition_fixed_effect_regression.csv", index=False)
regression_summary.style.format({"coef_codeonly_vs_baseline": "{:+.4f}", "std_err": "{:.4f}", "p_value": "{:.4f}", "rsquared": "{:.4f}"})

## 8. Error Category Comparison

This checks whether the system prompt changed failure modes.

In [ ]:
error_summary = (
    df.groupby(["condition", "error_category"])
    .size()
    .rename("runs")
    .reset_index()
)
error_summary["rate"] = error_summary["runs"] / error_summary.groupby("condition")["runs"].transform("sum")
error_summary.to_csv(OUTPUT_DIR / "error_category_summary.csv", index=False)
error_summary.sort_values(["condition", "runs"], ascending=[True, False]).style.format({"rate": "{:.1%}"})

## 8. Prompt-Factor Effects Within Each Condition

This section answers the key question:

> After forcing code-only output with a system prompt, do tone, role framing, and format still affect performance?

For each condition separately, we summarize performance/behavior by `Tone`, `Role_framing`, and `Format`. Then we run fixed-effect regressions within each condition:

```text
metric ~ Tone + Role_framing + Format + model fixed effects + problem fixed effects
```

This allows direct comparison of factor effects before and after adding the code-only system prompt.

In [ ]:
factor_metrics = ["passed_int", "tests_fraction", "completion_tokens", "duration_seconds", "likely_code_only", "extra_text_ratio", "invalid_python"]
factors = ["Tone", "Role_framing", "Format"]

condition_factor_tables = []
for condition in ["baseline_fixed", "codeonly_system"]:
    sub = df[df["condition"].eq(condition)].copy()
    for factor in factors:
        table = (
            sub.groupby(factor, dropna=False)
            .agg(
                runs=("task_id", "size"),
                pass_rate=("passed_int", "mean"),
                tests_fraction=("tests_fraction", "mean"),
                avg_completion_tokens=("completion_tokens", "mean"),
                avg_duration_seconds=("duration_seconds", "mean"),
                code_only_rate=("likely_code_only", "mean"),
                invalid_python_rate=("invalid_python", "mean"),
            )
            .reset_index()
        )
        table.insert(0, "condition", condition)
        table.insert(1, "factor", factor)
        condition_factor_tables.append(table)

condition_factor_summary = pd.concat(condition_factor_tables, ignore_index=True)
condition_factor_summary.to_csv(OUTPUT_DIR / "condition_factor_summary.csv", index=False)

for condition in ["baseline_fixed", "codeonly_system"]:
    print(f"\n=== {condition} ===")
    for factor in factors:
        print(f"\n-- {factor} --")
        display(
            condition_factor_summary[
                condition_factor_summary["condition"].eq(condition)
                & condition_factor_summary["factor"].eq(factor)
            ].style.format({
                "pass_rate": "{:.1%}",
                "tests_fraction": "{:.1%}",
                "avg_completion_tokens": "{:.0f}",
                "avg_duration_seconds": "{:.2f}",
                "code_only_rate": "{:.1%}",
                "invalid_python_rate": "{:.1%}",
            })
        )

In [ ]:
def factor_regressions_for_condition(condition: str) -> pd.DataFrame:
    sub = df[df["condition"].eq(condition)].copy()
    rows = []
    for metric in factor_metrics:
        formula = f"{metric} ~ C(Tone) + C(Role_framing) + C(Format) + C(model_short) + C(problem_id)"
        model_data = sub[[metric, "Tone", "Role_framing", "Format", "model_short", "problem_id"]].dropna()
        result = smf.ols(formula, data=model_data).fit(cov_type="HC3")
        for term, coef in result.params.items():
            if "Tone" in term or "Role_framing" in term or "Format" in term:
                rows.append({
                    "condition": condition,
                    "metric": metric,
                    "term": term,
                    "coef": coef,
                    "std_err": result.bse[term],
                    "p_value": result.pvalues[term],
                    "nobs": result.nobs,
                })
    return pd.DataFrame(rows)

factor_regression = pd.concat([
    factor_regressions_for_condition("baseline_fixed"),
    factor_regressions_for_condition("codeonly_system"),
], ignore_index=True)
factor_regression.to_csv(OUTPUT_DIR / "condition_factor_fixed_effect_regression.csv", index=False)

# Show correctness-related coefficients first.
correctness_terms = factor_regression[factor_regression["metric"].isin(["passed_int", "tests_fraction"])].copy()
display(correctness_terms.style.format({"coef": "{:+.4f}", "std_err": "{:.4f}", "p_value": "{:.4f}"}))

In [ ]:
# Compact side-by-side table for the most important factor coefficients.
important_terms = [
    "C(Tone)[T.neutral]",
    "C(Tone)[T.polite]",
    "C(Role_framing)[T.no role]",
    "C(Format)[T.plaintext]",
    "C(Format)[T.xml]",
]
compact = factor_regression[
    factor_regression["metric"].isin(["passed_int", "tests_fraction", "completion_tokens", "likely_code_only"])
    & factor_regression["term"].isin(important_terms)
].copy()
compact_pivot = compact.pivot_table(index=["metric", "term"], columns="condition", values=["coef", "p_value"], aggfunc="first")
compact.to_csv(OUTPUT_DIR / "condition_factor_key_terms.csv", index=False)
compact_pivot

## Visual Summary: Overall Before vs Code-Only

These charts show the high-level tradeoff: code-only greatly improves compliance and speed, while performance improves more modestly.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
visual_metrics = [
    ("pass_rate", "Pass rate", "percent"),
    ("tests_fraction", "Avg. tests passed", "percent"),
    ("avg_duration_seconds", "Avg. duration (s)", "number"),
    ("code_only_rate", "Code-only rate", "percent"),
]
for ax, (metric, title, kind) in zip(axes, visual_metrics):
    sns.barplot(data=summary, x="condition", y=metric, ax=ax, hue="condition", legend=False)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
    if kind == "percent":
        ax.set_ylim(0, max(summary[metric].max() * 1.25, 0.05))
        ax.yaxis.set_major_formatter(lambda x, pos: f"{100*x:.0f}%")
    for container in ax.containers:
        labels = [f"{100*v.get_height():.1f}%" if kind == "percent" else f"{v.get_height():.1f}" for v in container]
        ax.bar_label(container, labels=labels, fontsize=9, padding=2)
plt.tight_layout()
plt.show()

## Visual Summary: Model-Level Changes

These charts show that the code-only system prompt helps Gemma and Llama more than Qwen on exact pass rate, while all models become much more code-only and shorter.

In [ ]:
model_delta_long = model_deltas.melt(
    id_vars="model_short",
    value_vars=["delta_passed_int_mean", "delta_tests_fraction_mean", "delta_completion_tokens_mean", "delta_duration_seconds_mean", "delta_likely_code_only_mean"],
    var_name="metric",
    value_name="delta",
)
label_map = {
    "delta_passed_int_mean": "Pass-rate delta",
    "delta_tests_fraction_mean": "Tests-fraction delta",
    "delta_completion_tokens_mean": "Token delta",
    "delta_duration_seconds_mean": "Duration delta",
    "delta_likely_code_only_mean": "Code-only-rate delta",
}
model_delta_long["metric_label"] = model_delta_long["metric"].map(label_map)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title, percent in [
    (axes[0], "delta_passed_int_mean", "Pass-rate delta", True),
    (axes[1], "delta_tests_fraction_mean", "Tests-fraction delta", True),
    (axes[2], "delta_completion_tokens_mean", "Completion-token delta", False),
]:
    plot_data = model_deltas.copy()
    sns.barplot(data=plot_data, x="model_short", y=metric, ax=ax, color="#4C78A8")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
    if percent:
        ax.yaxis.set_major_formatter(lambda x, pos: f"{100*x:+.0f} pp")
        for container in ax.containers:
            ax.bar_label(container, labels=[f"{100*v.get_height():+.1f} pp" for v in container], fontsize=9, padding=2)
    else:
        for container in ax.containers:
            ax.bar_label(container, labels=[f"{v.get_height():+.0f}" for v in container], fontsize=9, padding=2)
plt.tight_layout()
plt.show()

## Visual Summary: Tone Effects Under Each Condition

These charts answer: after forcing code-only output, does tone still matter?

The answer from the current data is yes: neutral remains strongest, but the gap is smaller than before.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in [
    (axes[0], "tests_fraction", "Avg. tests passed"),
    (axes[1], "avg_completion_tokens", "Completion tokens"),
    (axes[2], "code_only_rate", "Code-only rate"),
]:
    plot_data = condition_factor_summary[condition_factor_summary["factor"].eq("Tone")].copy()
    sns.barplot(data=plot_data, x="Tone", y=metric, hue="condition", ax=ax, order=["neutral", "polite", "aggressive"])
    ax.set_title(title)
    ax.set_xlabel("")
    if metric in ["tests_fraction", "code_only_rate"]:
        ax.yaxis.set_major_formatter(lambda x, pos: f"{100*x:.0f}%")
plt.tight_layout()
plt.show()

## Heatmap: Model × Tone Deltas

This heatmap shows where the code-only system prompt helped or hurt. Values are `codeonly_system - baseline_fixed`.

In [ ]:
heat_metrics = [
    ("delta_tests_fraction_mean", "Tests-fraction delta"),
    ("delta_completion_tokens_mean", "Completion-token delta"),
    ("delta_likely_code_only_mean", "Code-only-rate delta"),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (metric, title) in zip(axes, heat_metrics):
    pivot = model_tone_deltas.pivot(index="model_short", columns="Tone", values=metric)[["neutral", "polite", "aggressive"]]
    fmt = ".2f" if "tokens" not in metric else ".0f"
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap="RdBu_r", center=0, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

## Factor Effect Coefficients Before vs Code-Only

This chart visualizes the fixed-effect regression coefficients for correctness. It shows which prompt factors remain meaningful under the code-only system prompt.

In [ ]:
coef_plot = factor_regression[
    factor_regression["metric"].isin(["passed_int", "tests_fraction"])
    & factor_regression["term"].isin([
        "C(Tone)[T.neutral]", "C(Tone)[T.polite]",
        "C(Role_framing)[T.no role]",
        "C(Format)[T.plaintext]", "C(Format)[T.xml]",
    ])
].copy()
coef_plot["ci_low"] = coef_plot["coef"] - 1.96 * coef_plot["std_err"]
coef_plot["ci_high"] = coef_plot["coef"] + 1.96 * coef_plot["std_err"]
coef_plot["term_label"] = coef_plot["term"].replace({
    "C(Tone)[T.neutral]": "Tone: neutral",
    "C(Tone)[T.polite]": "Tone: polite",
    "C(Role_framing)[T.no role]": "Role: no role",
    "C(Format)[T.plaintext]": "Format: plaintext",
    "C(Format)[T.xml]": "Format: XML",
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, metric, title in [(axes[0], "passed_int", "Pass/fail coefficient"), (axes[1], "tests_fraction", "Tests-fraction coefficient")]:
    sub = coef_plot[coef_plot["metric"].eq(metric)].copy()
    y_order = ["Tone: neutral", "Tone: polite", "Role: no role", "Format: plaintext", "Format: XML"]
    offsets = {"baseline_fixed": -0.12, "codeonly_system": 0.12}
    for condition, color in [("baseline_fixed", "#4C78A8"), ("codeonly_system", "#F58518")]:
        csub = sub[sub["condition"].eq(condition)].copy()
        y = np.array([y_order.index(v) for v in csub["term_label"]]) + offsets[condition]
        ax.errorbar(
            csub["coef"], y,
            xerr=[csub["coef"] - csub["ci_low"], csub["ci_high"] - csub["coef"]],
            fmt="o", capsize=3, label=condition, color=color,
        )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(range(len(y_order)))
    ax.set_yticklabels(y_order)
    ax.set_title(title)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{100*x:+.0f} pp")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

## 9. Write Short Markdown Summary

This writes a compact report to `analysis_outputs/codeonly_comparison/codeonly_comparison_report.md`.

In [ ]:
def pct(x): return f"{100*x:.1f}%"
def pp(x): return f"{100*x:+.1f} pp"

over = summary.set_index("condition")
base = over.loc["baseline_fixed"]
co = over.loc["codeonly_system"]

def model_line(model):
    b = model_summary[(model_summary.model_short.eq(model)) & (model_summary.condition.eq("baseline_fixed"))].iloc[0]
    c = model_summary[(model_summary.model_short.eq(model)) & (model_summary.condition.eq("codeonly_system"))].iloc[0]
    return f"| {model} | {int(b.passes)}/900 ({pct(b.pass_rate)}) | {int(c.passes)}/900 ({pct(c.pass_rate)}) | {pp(c.pass_rate - b.pass_rate)} | {pp(c.tests_fraction - b.tests_fraction)} | {c.avg_completion_tokens - b.avg_completion_tokens:+.0f} | {pp(c.code_only_rate - b.code_only_rate)} |"

report = f"""# Code-Only System Prompt Comparison

System prompt tested:

> Output only the code, no explanations.

Baseline: corrected fixed-extractor results (`results/*_fixextractor.jsonl`).
Code-only condition: `results_codeonly/*.jsonl`.

## Overall Result

| Condition | Passes | Pass rate | Avg. tests passed | Avg. completion tokens | Avg. duration | Code-only rate |
|---|---:|---:|---:|---:|---:|---:|
| baseline_fixed | {int(base.passes)}/2700 | {pct(base.pass_rate)} | {pct(base.tests_fraction)} | {base.avg_completion_tokens:.0f} | {base.avg_duration_seconds:.2f}s | {pct(base.code_only_rate)} |
| codeonly_system | {int(co.passes)}/2700 | {pct(co.pass_rate)} | {pct(co.tests_fraction)} | {co.avg_completion_tokens:.0f} | {co.avg_duration_seconds:.2f}s | {pct(co.code_only_rate)} |

Overall, the code-only system prompt changed exact passes from {int(base.passes)}/2700 to {int(co.passes)}/2700, a pass-rate change of {pp(co.pass_rate - base.pass_rate)}. It changed average tests passed by {pp(co.tests_fraction - base.tests_fraction)}, completion length by {co.avg_completion_tokens - base.avg_completion_tokens:+.0f} tokens, and code-only rate by {pp(co.code_only_rate - base.code_only_rate)}.

## By Model

| Model | Baseline passes | Code-only passes | Pass-rate delta | Tests-fraction delta | Token delta | Code-only-rate delta |
|---|---:|---:|---:|---:|---:|---:|
"""
for model in sorted(model_summary.model_short.unique()):
    report += model_line(model) + "\n"

report += """

## Interpretation

The system prompt improves measured exact pass rate overall, but the effect is model-specific. It helps Gemma and Llama, while Qwen's exact pass count is unchanged. The comparison should be interpreted as a controlled prompt-policy intervention: adding a global code-only instruction changes output behavior and modestly changes downstream performance.

The key research implication is that explicit output constraints can reduce evaluator-compliance problems, but they do not remove prompt-factor effects entirely. Under the code-only system prompt, tone still affects performance: neutral remains best overall by tests fraction, while polite/aggressive improve more than neutral because they had the largest verbosity/compliance problem in the baseline. Role framing remains small, and format effects remain modest/model-dependent.
"""

report_path = OUTPUT_DIR / "codeonly_comparison_report.md"
report_path.write_text(report, encoding="utf-8")
print(report_path)
print(report)